# Predicting Irrigation Need

In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

from imblearn.over_sampling import SMOTE

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier # GradientBoostingClassifier: not good for imbalanced data
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
# from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import classification_report, balanced_accuracy_score

## Load dataset

In [ ]:
train = pd.read_csv('./playground-series-s6e4/train.csv')
test = pd.read_csv('./playground-series-s6e4/test.csv')
# train.head()

In [ ]:
train.columns

In [ ]:
# test.tail()

In [ ]:
# train.info()

In [ ]:
# train.describe()

In [ ]:
# train.isna().sum()

## Cleaning data for EDA:
- Drop column id


In [ ]:
test_id = test['id']
for data in [train, test]:
	data.drop(columns=['id'], inplace=True)

- Any duplicated row?

In [ ]:
train.duplicated().sum()

- How imbalaced is the classes?

In [ ]:
plt.figure(figsize=(10, 4))
plt.pie(
    train['Irrigation_Need'].value_counts(),
    labels=train['Irrigation_Need'].value_counts().index,
    autopct='%1.1f%%'
)
plt.title(f"Distribution of Labels")
plt.show()

- Cheking the outliers:

In [ ]:
num_cols = train.select_dtypes(include=np.number).columns
fig, axes = plt.subplots(4, 3, figsize=(18, 15))
axes = axes.flatten()
for i, col in enumerate(num_cols):
	sns.boxplot(train[col], ax=axes[i], orient='h')
	axes[i].set_title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

- Pair plot takes a lot of time ...

In [ ]:
# corr = train.corr(numeric_only=True)
# figure, ax = plt.subplots(figsize=(10, 8))
# sns.heatmap(ax=ax, data=corr, annot=True, cmap='coolwarm')

## Cleaning data for modelling

In [ ]:
X = train.drop(columns=['Irrigation_Need'])
Y = train['Irrigation_Need']

In [ ]:
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

In [ ]:
le = LabelEncoder()
Y_train_enc = le.fit_transform(Y_train)
Y_val_enc = le.transform(Y_val)

In [ ]:
cat_cols = [col for col in X_train.columns if X_train[col].dtype == 'object']
num_cols = [col for col in X_train.columns if X_train[col].dtype != 'object']

In [ ]:
preprocessor_tree = ColumnTransformer(
	transformers=[
		('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_cols),
	],
	remainder='passthrough'
)

preprocessor_linear = ColumnTransformer(
	transformers=[
		('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_cols),
		('num', StandardScaler(), num_cols)
	],
	remainder='passthrough'
)

pipe = Pipeline([
	('preprocessor', preprocessor_linear), # placeholder, will be set in GridSearchCV
    ('smote', SMOTE(random_state=42)), # placeholder, will be set in GridSearchCV
	('model', LogisticRegression()) # placeholder, will be set in GridSearchCV
])

In [ ]:
param_grid = [

    # 🌲 Random Forest (no scaling)
    {
        'preprocessor': [preprocessor_tree],
        'model': [RandomForestClassifier()],
        'model__n_estimators': [50, 100],
        'model__max_depth': [None, 10, 20],
        'model__class_weight': ['balanced']
    },

    # 🌳 Decision Tree (no scaling)
    {
        'preprocessor': [preprocessor_tree],
        'model': [DecisionTreeClassifier()],
        'model__max_depth': [None, 10, 20],
		'model__class_weight': ['balanced']
    },

    # 📈 Logistic Regression (needs scaling)
    # {
    #     'preprocessor': [preprocessor_linear],
    #     'model': [LogisticRegression(max_iter=1000)],
    #     'model__C': [0.1, 1, 10]
    # },

    # # 🧠 SVM (needs scaling)
    # {
    #     'preprocessor': [preprocessor_linear],
    #     'model': [SVC()],
    #     'model__C': [0.1, 1, 10],
    #     'model__kernel': ['rbf', 'linear']
    # }
    # {
    #     'preprocessor': [preprocessor_linear],
	# 	'model': [GradientBoostingClassifier()],
	# 	'model__n_estimators': [50, 100],
	# 	'model__learning_rate': [0.01, 0.1],
	# }
{
    'preprocessor': [preprocessor_linear],
    'model': [XGBClassifier(
        objective='multi:softprob',  # for multiclass
        eval_metric='mlogloss',
        use_label_encoder=False
    )],
    'model__n_estimators': [100, 200],
    'model__max_depth': [3, 6],
    'model__learning_rate': [0.01, 0.1],
    'model__subsample': [0.8, 1.0]
},

{
    'preprocessor': [preprocessor_linear],
    'model': [LGBMClassifier()],
    'model__n_estimators': [100, 200],
    'model__max_depth': [-1, 10],
    'model__learning_rate': [0.01, 0.1],
    'model__num_leaves': [31, 63]
}
]

scoring={
	'f1_macro': 'f1_macro',
	'roc_auc': 'roc_auc_ovr',
	'precision': 'precision_macro'
}

grid = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring=scoring,
    refit='f1_macro',
    # n_jobs=-1,
    verbose=2
)

grid.fit(X_train, Y_train_enc)


- Save the grid search result for later use, only the best model:

In [ ]:
best_model = grid.best_estimator_
import joblib
joblib.dump(best_model, 'best_model.pkl')
# best_model = joblib.load('best_model.pkl')

- Save all of the result:

In [ ]:
joblib.dump(grid, 'grid_search.pkl') # It could be large!
# grid = joblib.load('grid_search.pkl')

In [ ]:
print(f"Best Parameters: {grid.best_params_}")

In [ ]:
best_pipeline = grid.best_estimator_
print(f"Best pipeline steps: {best_pipeline.named_steps}")
Y_pred_enc = best_pipeline.predict(X_val)
print(classification_report(Y_val_enc, Y_pred_enc))
print(f"Balanced Accuracy Score: {balanced_accuracy_score(Y_val_enc, Y_pred_enc):.4f}")

DT:  
Best Parameters: {'criterion': 'entropy', 'max_depth': 15, 'min_samples_leaf': 4, 'min_samples_split': 10}

In [ ]:
# model = DecisionTreeClassifier(
#     criterion=grid_search.best_params_['criterion'],
#     max_depth=grid_search.best_params_['max_depth'],
#     min_samples_split=grid_search.best_params_['min_samples_split'],
#     min_samples_leaf=grid_search.best_params_['min_samples_leaf'],
#     random_state=42)
# model.fit(X_train, Y_train)
# Y_pred = model.predict(X_val)
# print(classification_report(Y_val, Y_pred))
# print(f"Balanced Accuracy Score: {balanced_accuracy_score(Y_val, Y_pred):.4f}")

### Train on all data before prediction:

In [ ]:
X_tv = pd.concat([X_train, X_val], axis=0)
Y_tv = np.concatenate((Y_train_enc, Y_val_enc), axis=0)

best_pipeline.fit(X_tv, Y_tv)

In [ ]:
prediction_enc = best_pipeline.predict(test)
prediction_decoded = le.inverse_transform(prediction_enc)
output = pd.DataFrame({'id': test_id, 'Irrigation_Need': prediction_decoded})
output.head()

In [ ]:
output.to_csv('submission7.csv', index=False)